# ARC Phase 2 Arbitration Policy Pipeline

This notebook launches the full phase-2 pipeline: SFT with `accelerate`, GRPO with `accelerate`, validation on `data/val.jsonl`, and final testing on `data/test.jsonl`. The notebook does not create validation or test splits from the GRPO training file.

## 1. Runtime Parameters

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import torch

REPO_URL = "https://github.com/beryl-07/arc.git"
PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR).endswith("/notebooks"):
    PROJECT_DIR = PROJECT_DIR.parent
if Path("/kaggle/working").exists():
    PROJECT_DIR = Path("/kaggle/working/content/arc")

FORCE_RECLONE = False
RUN_SFT = True
RUN_GRPO = True
RUN_VALIDATION = True
RUN_TEST = True

NUM_PROCESSES = max(1, torch.cuda.device_count())
USE_MULTI_GPU = NUM_PROCESSES > 1
ACCELERATE_MIXED_PRECISION = "no"

print("Project:", PROJECT_DIR)
print("GPUs:", NUM_PROCESSES)
print("Accelerate mixed precision:", ACCELERATE_MIXED_PRECISION)

## 2. Repository and Dependencies

In [ ]:
if Path("/kaggle/working").exists():
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "git", "git-lfs", "wget"], check=True)
    subprocess.run(["git", "lfs", "install"], check=True)

    if FORCE_RECLONE and PROJECT_DIR.exists():
        import shutil
        shutil.rmtree(PROJECT_DIR)
    if PROJECT_DIR.exists():
        subprocess.run(["git", "pull"], cwd=PROJECT_DIR, check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

# Valeurs d'environnement minimales pour un lancement autonome du notebook.
# Elles sont posées avant l'import de src.config, qui lit l'environnement à l'import.
os.environ.setdefault("SFT_VAL_PATH", "outputs/sft_val_messages.jsonl")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
sys.path.insert(0, str(PROJECT_DIR))
os.environ["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
from src.config import ARBITRATION_CONFIG

for module_name in ["torch", "transformers", "trl", "peft", "datasets", "sklearn", "accelerate"]:
    module = __import__(module_name)
    print(f"{module_name}: {getattr(module, '__version__', 'unknown')}")

# Les valeurs ci-dessous proviennent de la configuration centrale.
MODEL_NAME = ARBITRATION_CONFIG.model_name
SFT_TRAIN_PATH = PROJECT_DIR / ARBITRATION_CONFIG.sft_train_path
SFT_VAL_PATH = PROJECT_DIR / ARBITRATION_CONFIG.sft_val_path if ARBITRATION_CONFIG.sft_val_path else None
GRPO_TRAIN_PATH = PROJECT_DIR / ARBITRATION_CONFIG.grpo_train_path
VAL_PATH = PROJECT_DIR / ARBITRATION_CONFIG.val_path
TEST_PATH = PROJECT_DIR / ARBITRATION_CONFIG.test_path
SFT_OUTPUT_DIR = PROJECT_DIR / ARBITRATION_CONFIG.sft_output_dir
GRPO_OUTPUT_DIR = PROJECT_DIR / ARBITRATION_CONFIG.grpo_output_dir
OUTPUTS_DIR = PROJECT_DIR / ARBITRATION_CONFIG.outputs_dir
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print("Ready:", PROJECT_DIR)

## 3. Preflight Data Check

In [ ]:
import json
from src.arbitration_policy import build_arbitration_prompt

def build_sft_validation_file(source_path, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    rows = []
    with source_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            record = json.loads(line)
            prompt_messages = build_arbitration_prompt(record)
            gold = record.get("gold_answer", [])
            response = gold if isinstance(gold, list) else [gold]
            assistant = {
                "role": "assistant",
                "content": json.dumps({
                    "strategy": "validation",
                    "response": response,
                    "justification": "Reference answer from the held-out validation split."
                }, ensure_ascii=False),
            }
            rows.append({"messages": prompt_messages + [assistant]})
    with output_path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"Built SFT validation file: {output_path} ({len(rows)} rows)")

if RUN_SFT and SFT_VAL_PATH is None:
    raise ValueError("SFT_VAL_PATH is mandatory for SFT training and must point to a conversational JSONL file with `messages`.")
if RUN_SFT and SFT_VAL_PATH is not None and not SFT_VAL_PATH.exists():
    build_sft_validation_file(VAL_PATH, SFT_VAL_PATH)

paths = [SFT_TRAIN_PATH, GRPO_TRAIN_PATH, VAL_PATH, TEST_PATH]
if SFT_VAL_PATH is not None:
    paths.append(SFT_VAL_PATH)

for path in paths:
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open("r", encoding="utf-8") as handle:
        n = sum(1 for line in handle if line.strip())
    print(f"{path.relative_to(PROJECT_DIR)}: {n} rows")

with SFT_TRAIN_PATH.open("r", encoding="utf-8") as handle:
    first_sft = json.loads(next(handle))
if "messages" not in first_sft:
    raise ValueError("SFT training file must contain conversational `messages` records.")

## 4. SFT Training

In [ ]:
def run_checked(cmd):
    print(" ".join(cmd))
    completed = subprocess.run(
        cmd,
        cwd=PROJECT_DIR,
        env=os.environ.copy(),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if completed.stdout:
        print("--- STDOUT ---")
        print(completed.stdout)
    if completed.stderr:
        print("--- STDERR ---")
        print(completed.stderr)
    completed.check_returncode()
    return completed

def accelerate_prefix():
    cmd = ["accelerate", "launch", "--num_processes", str(NUM_PROCESSES), "--mixed_precision", ACCELERATE_MIXED_PRECISION]
    if USE_MULTI_GPU:
        cmd.insert(2, "--multi_gpu")
    return cmd

if RUN_SFT:
    cmd = accelerate_prefix() + [
        str(PROJECT_DIR / "scripts" / "train_sft_arbitration.py"),
        "--model-name", MODEL_NAME,
        "--train-path", str(SFT_TRAIN_PATH),
        "--val-path", str(SFT_VAL_PATH),
        "--output-dir", str(SFT_OUTPUT_DIR),
        "--epochs", str(ARBITRATION_CONFIG.sft_num_epochs),
        "--learning-rate", str(ARBITRATION_CONFIG.sft_learning_rate),
        "--max-length", str(ARBITRATION_CONFIG.sft_max_seq_length),
        "--save-steps", str(ARBITRATION_CONFIG.sft_save_steps),
        "--per-device-train-batch-size", str(ARBITRATION_CONFIG.sft_per_device_train_batch_size),
        "--gradient-accumulation-steps", str(ARBITRATION_CONFIG.sft_gradient_accumulation_steps),
    ]
    run_checked(cmd)
else:
    print("Skipping SFT.")

## 5. GRPO Training

In [ ]:
if RUN_GRPO:
    cmd = accelerate_prefix() + [
        str(PROJECT_DIR / "scripts" / "train_grpo_arbitration.py"),
        "--sft-model-path", str(SFT_OUTPUT_DIR),
        "--train-path", str(GRPO_TRAIN_PATH),
        "--output-dir", str(GRPO_OUTPUT_DIR),
        "--num-generations", str(ARBITRATION_CONFIG.grpo_num_generations),
        "--max-steps", str(ARBITRATION_CONFIG.grpo_max_steps),
        "--learning-rate", str(ARBITRATION_CONFIG.grpo_learning_rate),
        "--beta", str(ARBITRATION_CONFIG.grpo_beta),
        "--max-completion-length", str(ARBITRATION_CONFIG.grpo_max_completion_length),
        "--per-device-train-batch-size", str(ARBITRATION_CONFIG.grpo_per_device_batch_size),
        "--gradient-accumulation-steps", str(ARBITRATION_CONFIG.grpo_gradient_accumulation_steps),
    ]
    run_checked(cmd)
else:
    print("Skipping GRPO.")

## 6. Validation and Test

In [ ]:
def run_eval(split_name, data_path):
    output_path = OUTPUTS_DIR / f"arbitration_{split_name}_predictions.json"
    cmd = [
        sys.executable,
        str(PROJECT_DIR / "scripts" / "evaluate_arbitration_policy.py"),
        "--model-path", str(GRPO_OUTPUT_DIR),
        "--data-path", str(data_path),
        "--output-path", str(output_path),
    ]
    run_checked(cmd)

if RUN_VALIDATION:
    run_eval("val", VAL_PATH)
if RUN_TEST:
    run_eval("test", TEST_PATH)